In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import ast

# ===================== Config =====================
sc_path   = r"F:\Thesis\project\1-BaselineTest\Models Responses\self-consistency\llama 4 maverick\results_llama4-maverick.csv"  # self-consistency CSV
gold_path = r"F:\Thesis\project\1-BaselineTest\GOLD\Gold.csv"  # cols: idx, Gold

# نام ستون ID در فایل‌ها
model_id_col = "id"
gold_id_col  = "idx"
gold_ans_col = "Gold"

# روت خروجی
out_root = Path("eval_sc")
out_root.mkdir(parents=True, exist_ok=True)

# ===================== Plot style =====================
sns.set_style("whitegrid")
plt.rcParams["font.family"] = "Arial"

def save_plot(path):
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight", facecolor="white")
    plt.close()

# ===================== Helpers =====================
def norm_ans(x):
    """نرمال‌سازی جواب‌ها به رشته‌های '1','2','3','4' یا None."""
    if pd.isna(x):
        return None
    s = str(x).strip().strip('"').strip("'")
    try:
        v = int(float(s))
        return str(v) if v in {1, 2, 3, 4} else None
    except Exception:
        # اگر ته رشته یک رقم 1..4 باشد
        for d in ["1", "2", "3", "4"]:
            if s.endswith(d):
                return d
        return None

def parse_gold_multi(x):
    """پارس کردن جواب GOLD (ممکن است چندگزینه‌ای باشد مثل 1-3)."""
    if pd.isna(x):
        return set()
    s = str(x).strip()
    for sep in ['-', '/', '،', ',', '|']:
        if sep in s:
            parts = [p.strip() for p in s.split(sep)]
            return {norm_ans(p) for p in parts if norm_ans(p) is not None}
    v = norm_ans(s)
    return {v} if v is not None else set()

def parse_votes_map(s):
    """تبدیل رشته votes_map به دیکشنری {option: count} برای گزینه‌های 1..4."""
    if pd.isna(s):
        return {}
    try:
        d = ast.literal_eval(str(s))
        d = {str(k): int(v) for k, v in dict(d).items() if norm_ans(k) in {"1", "2", "3", "4"}}
        return d
    except Exception:
        return {}

def final_from_votes_map(d):
    """پاسخ نهایی از روی votes_map با plurality vote."""
    if not isinstance(d, dict) or not d:
        return None
    options = ["1", "2", "3", "4"]
    counts = {o: d.get(o, 0) for o in options}
    best = max(options, key=lambda o: counts[o])
    return best if counts[best] > 0 else None

def plurality_from_runs(row, run_cols):
    """پاسخ نهایی از روی plurality روی answer_run1..K."""
    votes = [row[c] for c in run_cols if c in row.index and pd.notna(row[c])]
    votes = [v for v in votes if v is not None]
    if not votes:
        return None
    options = ["1", "2", "3", "4"]
    counts = {o: votes.count(o) for o in options}
    best = max(options, key=lambda o: counts[o])
    return best if counts[best] > 0 else None

# ===================== Load =====================
df   = pd.read_csv(sc_path)
gold = pd.read_csv(gold_path)

# ===================== Detect columns =====================
run_cols  = [f"answer_run{i}"     for i in range(1, 6) if f"answer_run{i}"     in df.columns]
conf_cols = [f"confidence_run{i}" for i in range(1, 6) if f"confidence_run{i}" in df.columns]

# ===================== Normalize answers & confidences =====================
# نرمال‌سازی پاسخ‌های runs
for c in run_cols:
    df[c] = df[c].apply(norm_ans)

# votes_map → dict
if "votes_map" in df.columns:
    df["votes_map_dict"] = df["votes_map"].apply(parse_votes_map)
else:
    df["votes_map_dict"] = np.nan

# confidence‌ها به عدد
num_to_cast = ["confidence_mean", "confidence_vote", "latency_avg_ms", "total_time_ms"]
for c in conf_cols + num_to_cast:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# ===================== Build answer_final (اصلی‌ترین اصلاح) =====================
# 1) اگر answer_final در CSV است، به عنوان raw می‌خوانیم و نرمال می‌کنیم
if "answer_final" in df.columns:
    df["answer_final_raw"] = df["answer_final"].apply(norm_ans)
else:
    df["answer_final_raw"] = np.nan

# 2) final از votes_map
df["answer_final_from_votes"] = df["votes_map_dict"].apply(final_from_votes_map)

# 3) final از plurality روی runs
df["answer_final_from_runs"] = df.apply(lambda r: plurality_from_runs(r, run_cols), axis=1)

# 4) سیاست انتخاب نهایی:
#    اولویت: votes_map → plurality_runs → answer_final_raw
df["answer_final"] = df["answer_final_raw"]

mask_votes = df["answer_final_from_votes"].notna()
df.loc[mask_votes, "answer_final"] = df.loc[mask_votes, "answer_final_from_votes"]

mask_runs = df["answer_final"].isna() & df["answer_final_from_runs"].notna()
df.loc[mask_runs, "answer_final"] = df.loc[mask_runs, "answer_final_from_runs"]

# ===================== vote_ratio_max و consensus_max =====================
def vote_ratio(d):
    if not isinstance(d, dict) or not d:
        return np.nan
    tot = sum(d.values())
    return max(d.values()) / tot if tot > 0 else np.nan

df["vote_ratio_max"] = df["votes_map_dict"].apply(vote_ratio)

def consensus_from_votes_map(d):
    if not isinstance(d, dict) or not d:
        return 0
    return max(d.values())

df["consensus_max"] = df["votes_map_dict"].apply(consensus_from_votes_map)

df["consensus_label"] = df["consensus_max"].map(
    {5: "5/5", 4: "4/5", 3: "3/5", 2: "2/5", 1: "1/5", 0: "0/5"}
)

# ===================== Gold merge =====================
gold2 = gold.copy()
gold2["gold_set"]     = gold2[gold_ans_col].apply(parse_gold_multi)
gold2["gold_primary"] = gold2["gold_set"].apply(lambda s: sorted(list(s))[0] if len(s) > 0 else None)
gold2["is_multi"]     = gold2["gold_set"].apply(lambda s: len(s) > 1)

df = pd.merge(
    df,
    gold2[[gold_id_col, "gold_set", "gold_primary", "is_multi"]],
    left_on=model_id_col,
    right_on=gold_id_col,
    how="inner"
)

# ===================== Correctness =====================
df["correct"] = df.apply(
    lambda r: (r["answer_final"] in r["gold_set"]) if r["answer_final"] is not None else False,
    axis=1
).astype(int)

# ===================== Section 1: Overall accuracy =====================
sec = out_root / "1_eval_accuracy"
sec.mkdir(exist_ok=True)

acc = df["correct"].mean() * 100 if len(df) else 0.0
pd.DataFrame([{"accuracy_%": round(acc, 2), "n": len(df)}]) \
  .to_csv(sec / "accuracy_overall.csv", index=False, encoding="utf-8-sig")

acc_by_final = (
    df.groupby("answer_final")["correct"].mean().mul(100)
      .reset_index().rename(columns={"correct": "accuracy_%"})
)
acc_by_final.to_csv(sec / "accuracy_by_final.csv", index=False, encoding="utf-8-sig")

conf_mat = pd.crosstab(
    df["answer_final"],
    df["gold_primary"],
    rownames=["pred_final"],
    colnames=["gold_primary"],
    dropna=False
).fillna(0).astype(int)
conf_mat.to_csv(sec / "confusion_final_vs_gold.csv", encoding="utf-8-sig")

plt.figure(figsize=(4, 4))
plt.bar(["Accuracy"], [acc], color="#4E79A7")
plt.ylim(0, 100)
plt.ylabel("Accuracy (%)")
plt.title("Final Accuracy (Self-Consistency)")
plt.text(0, acc + 1, f"{acc:.1f}%", ha="center", va="bottom", fontweight="bold")
save_plot(sec / "plot_accuracy_overall.png")

plt.figure(figsize=(6, 4))
tmp = acc_by_final.copy()
tmp["answer_final"] = tmp["answer_final"].astype(str)
plt.bar(tmp["answer_final"], tmp["accuracy_%"], color="#F28E2B")
plt.ylim(0, 100)
plt.xlabel("Final answer")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy by final option")
for x, y in zip(tmp["answer_final"], tmp["accuracy_%"]):
    plt.text(x, y + 1, f"{y:.1f}%", ha="center", va="bottom", fontsize=9)
save_plot(sec / "plot_accuracy_by_final.png")

plt.figure(figsize=(5, 4))
sns.heatmap(conf_mat, annot=True, fmt="d", cmap="Blues", cbar=False,
            linewidths=1, linecolor="white")
plt.title("Confusion (final vs gold)")
save_plot(sec / "plot_confusion_final_vs_gold.png")

# ===================== Section 2: runs vs final =====================
sec = out_root / "2_runs_vs_final"
sec.mkdir(exist_ok=True)

acc_rows = [{"metric": "final", "accuracy_%": round(acc, 2), "n": len(df)}]
for i in range(1, 6):
    c = f"answer_run{i}"
    if c in df.columns:
        acc_i = (
            df.apply(
                lambda r: (r[c] in r["gold_set"]) if r[c] is not None else False,
                axis=1
            ).mean() * 100
        )
        acc_rows.append({"metric": f"run{i}", "accuracy_%": round(acc_i, 2), "n": len(df)})

acc_tbl = pd.DataFrame(acc_rows)
acc_tbl.to_csv(sec / "accuracy_runs_vs_final.csv", index=False, encoding="utf-8-sig")

plt.figure(figsize=(6, 4))
plt.bar(acc_tbl["metric"], acc_tbl["accuracy_%"], color="#76B7B2")
plt.ylim(0, 100)
plt.ylabel("Accuracy (%)")
plt.title("Accuracy: runs vs final (self-consistency gain)")
for x, y in zip(acc_tbl["metric"], acc_tbl["accuracy_%"]):
    plt.text(x, y + 1, f"{y:.1f}%", ha="center", va="bottom", fontsize=9)
save_plot(sec / "plot_accuracy_runs_vs_final.png")

# Confusion برای هر run
for i in range(1, 6):
    c = f"answer_run{i}"
    if c in df.columns:
        cm = pd.crosstab(
            df[c],
            df["gold_primary"],
            rownames=[f"pred_run{i}"],
            colnames=["gold_primary"],
            dropna=False
        ).fillna(0).astype(int)
        cm.to_csv(sec / f"confusion_run{i}_vs_gold.csv", encoding="utf-8-sig")
        plt.figure(figsize=(5, 4))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                    linewidths=1, linecolor="white")
        plt.title(f"Confusion (run{i} vs gold)")
        save_plot(sec / f"plot_confusion_run{i}_vs_gold.png")

# ===================== Section 3: consensus analysis =====================
sec = out_root / "3_consensus"
sec.mkdir(exist_ok=True)

acc_by_cons = (
    df.groupby("consensus_label")["correct"].mean().mul(100)
      .reset_index().rename(columns={"correct": "accuracy_%"})
)
cov_by_cons = (
    df["consensus_label"].value_counts(normalize=True)
      .mul(100).rename_axis("consensus_label").reset_index(name="coverage_%")
)
acc_by_cons.to_csv(sec / "accuracy_by_consensus.csv", index=False, encoding="utf-8-sig")
cov_by_cons.to_csv(sec / "coverage_by_consensus.csv", index=False, encoding="utf-8-sig")

dual = pd.merge(acc_by_cons, cov_by_cons, on="consensus_label", how="outer")
dual.to_csv(sec / "accuracy_coverage_consensus_dual.csv", index=False, encoding="utf-8-sig")

order = ["1/5", "2/5", "3/5", "4/5", "5/5"]
plt.figure(figsize=(6, 4))
acc_ordered = acc_by_cons.set_index("consensus_label").reindex(order)["accuracy_%"]
plt.bar(acc_ordered.index, acc_ordered.values, color="#4E79A7")
plt.ylim(0, 100)
plt.xlabel("Consensus (max votes / K)")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy vs Consensus")
save_plot(sec / "plot_accuracy_vs_consensus.png")

plt.figure(figsize=(6, 4))
cov_ordered = cov_by_cons.set_index("consensus_label").reindex(order)["coverage_%"]
plt.bar(cov_ordered.index, cov_ordered.values, color="#59A14F")
plt.ylim(0, 100)
plt.xlabel("Consensus (max votes / K)")
plt.ylabel("Coverage (%)")
plt.title("Coverage vs Consensus")
save_plot(sec / "plot_coverage_vs_consensus.png")

acc_grid = df.groupby(["consensus_label", "answer_final"])["correct"] \
             .mean().mul(100).unstack(fill_value=0)
acc_grid.to_csv(sec / "acc_grid_consensus_by_final.csv", encoding="utf-8-sig")
plt.figure(figsize=(6, 4))
sns.heatmap(acc_grid, annot=True, fmt=".1f", cmap="YlGnBu", cbar=True)
plt.title("Accuracy heatmap: consensus × final answer")
save_plot(sec / "plot_heatmap_acc_consensus_by_answer.png")

# ===================== Section 4: vote-ratio & reliability =====================
sec = out_root / "4_votes"
sec.mkdir(exist_ok=True)

if df["vote_ratio_max"].notna().any():
    bins   = [0, 0.6, 0.7, 0.8, 0.9, 1.0]
    labels = [f"{bins[i]}-{bins[i+1]}" for i in range(len(bins) - 1)]
    df["vote_ratio_bucket"] = pd.cut(df["vote_ratio_max"], bins=bins,
                                     labels=labels, include_lowest=True)

    acc_by_ratio = (
        df.groupby("vote_ratio_bucket")["correct"].mean().mul(100)
          .reset_index().rename(columns={"correct": "accuracy_%"})
    )
    acc_by_ratio.to_csv(sec / "accuracy_by_vote_ratio.csv", index=False, encoding="utf-8-sig")

    # counts per option
    def count_votes_row(row):
        votes = []
        for c in run_cols:
            v = row.get(c)
            if v is not None:
                votes.append(v)
        return pd.Series({a: votes.count(a) for a in ["1", "2", "3", "4"]})

    vote_counts = df.apply(count_votes_row, axis=1)
    vc = vote_counts.copy()
    vc.columns = [f"count_{c}" for c in vc.columns]
    vc.insert(0, model_id_col, df[model_id_col].values)
    vc["vote_ratio_max"] = df["vote_ratio_max"]
    vc.to_csv(sec / "parsed_votes_counts.csv", index=False, encoding="utf-8-sig")

    # Histogram vote_ratio_max
    plt.figure(figsize=(6, 4))
    plt.hist(df["vote_ratio_max"].dropna(), bins=np.linspace(0, 1, 21),
             color="#EDC948", edgecolor="black")
    plt.xlabel("Vote ratio max")
    plt.ylabel("Freq")
    plt.title("Vote ratio distribution")
    save_plot(sec / "plot_vote_ratio_hist.png")

    # Accuracy vs vote_ratio bucket
    plt.figure(figsize=(6, 4))
    plt.bar(acc_by_ratio["vote_ratio_bucket"].astype(str),
            acc_by_ratio["accuracy_%"], color="#F28E2B")
    plt.ylim(0, 100)
    plt.xlabel("Vote ratio bucket")
    plt.ylabel("Accuracy (%)")
    plt.title("Accuracy vs vote ratio")
    save_plot(sec / "plot_accuracy_vs_vote_ratio.png")

    # Reliability curve برای vote_ratio_max (binning fine)
    bins_rel = np.linspace(0, 1, 11)
    df["vote_ratio_bin"] = pd.cut(df["vote_ratio_max"], bins=bins_rel, include_lowest=True)
    rel_vr = df.groupby("vote_ratio_bin")["correct"].mean().mul(100) \
               .reset_index().rename(columns={"correct": "accuracy_%"})
    rel_vr.to_csv(sec / "reliability_vote_ratio.csv", index=False, encoding="utf-8-sig")

    plt.figure(figsize=(5, 4))
    xs = [interval.mid for interval in rel_vr["vote_ratio_bin"].cat.categories]
    ys = rel_vr.set_index("vote_ratio_bin")["accuracy_%"] \
                .reindex(rel_vr["vote_ratio_bin"].cat.categories).values
    plt.plot(xs, ys, marker="o", color="#1f77b4")
    plt.xlabel("vote_ratio_max (bin center)")
    plt.ylabel("Accuracy (%)")
    plt.ylim(0, 100)
    plt.title("Reliability: vote_ratio_max")
    save_plot(sec / "plot_reliability_vote_ratio.png")

# ===================== Section 5: calibration (confidence) =====================
sec = out_root / "5_calibration"
sec.mkdir(exist_ok=True)

# calibration confidence_mean (فرض: مقیاس 0..100 یا 0..X → نرمال‌سازی)
if "confidence_mean" in df.columns and df["confidence_mean"].notna().any():
    # نرمال‌سازی به [0,1] فرض می‌کنیم max=100
    df["conf_mean_norm"] = df["confidence_mean"] / df["confidence_mean"].max()
    bins = np.linspace(0, 1, 11)
    df["conf_mean_bin"] = pd.cut(df["conf_mean_norm"], bins=bins, include_lowest=True)
    calib = df.groupby("conf_mean_bin")["correct"].mean().mul(100) \
              .reset_index().rename(columns={"correct": "accuracy_%"})
    calib.to_csv(sec / "calibration_confidence_mean_binned.csv", index=False, encoding="utf-8-sig")

    plt.figure(figsize=(5, 4))
    xs = [interval.mid for interval in calib["conf_mean_bin"].cat.categories]
    ys = calib.set_index("conf_mean_bin")["accuracy_%"] \
              .reindex(calib["conf_mean_bin"].cat.categories).values
    plt.plot(xs, ys, marker="o", color="#4E79A7")
    plt.ylim(0, 100)
    plt.xlabel("confidence_mean (normalized, bin center)")
    plt.ylabel("Accuracy (%)")
    plt.title("Reliability (confidence_mean)")
    save_plot(sec / "plot_reliability_conf_mean_binned.png")

    # threshold tradeoff (مثلا روی scale خام)
    rows = []
    thresholds = np.linspace(df["confidence_mean"].min(), df["confidence_mean"].max(), 6)
    for t in thresholds:
        sub = df[df["confidence_mean"] >= t]
        rows.append({
            "threshold": round(t, 2),
            "coverage_%": (len(sub) / len(df) * 100) if len(df) else 0.0,
            "accuracy_%": (sub["correct"].mean() * 100) if len(sub) else np.nan,
            "n": len(sub)
        })
    trade = pd.DataFrame(rows)
    trade.to_csv(sec / "threshold_tradeoff_conf_mean.csv", index=False, encoding="utf-8-sig")

    plt.figure(figsize=(6, 4))
    ax1 = plt.gca()
    ax1.plot(trade["threshold"], trade["accuracy_%"], marker="o",
             color="#F28E2B", label="Accuracy")
    ax1.set_ylabel("Accuracy (%)", color="#F28E2B")
    ax1.set_xlabel("confidence_mean threshold")
    ax1.set_ylim(0, 100)

    ax2 = ax1.twinx()
    ax2.plot(trade["threshold"], trade["coverage_%"], marker="s",
             color="#59A14F", label="Coverage")
    ax2.set_ylabel("Coverage (%)", color="#59A14F")
    ax2.set_ylim(0, 100)

    plt.title("Accuracy vs Coverage by confidence_mean threshold")
    save_plot(sec / "plot_accuracy_coverage_conf_mean_threshold.png")

# calibration confidence_vote
if "confidence_vote" in df.columns and df["confidence_vote"].notna().any():
    df["conf_vote_norm"] = df["confidence_vote"] / df["confidence_vote"].max()
    bins = np.linspace(0, 1, 11)
    df["conf_vote_bin"] = pd.cut(df["conf_vote_norm"], bins=bins, include_lowest=True)
    calib_v = df.groupby("conf_vote_bin")["correct"].mean().mul(100) \
               .reset_index().rename(columns={"correct": "accuracy_%"})
    calib_v.to_csv(sec / "calibration_confidence_vote_binned.csv", index=False, encoding="utf-8-sig")

    plt.figure(figsize=(5, 4))
    xs = [interval.mid for interval in calib_v["conf_vote_bin"].cat.categories]
    ys = calib_v.set_index("conf_vote_bin")["accuracy_%"] \
                .reindex(calib_v["conf_vote_bin"].cat.categories).values
    plt.plot(xs, ys, marker="o", color="#76B7B2")
    plt.ylim(0, 100)
    plt.xlabel("confidence_vote (normalized, bin center)")
    plt.ylabel("Accuracy (%)")
    plt.title("Reliability (confidence_vote)")
    save_plot(sec / "plot_reliability_conf_vote_binned.png")

# heatmap accuracy by conf_mean bucket × final answer
if "confidence_mean" in df.columns:
    bins = [0, 20, 40, 60, 80, 100]
    df["conf_mean_bucket"] = pd.cut(df["confidence_mean"], bins=bins,
                                    labels=[f"{bins[i]}-{bins[i+1]}" for i in range(len(bins)-1)])
    grid = df.groupby(["conf_mean_bucket", "answer_final"])["correct"] \
             .mean().mul(100).unstack(fill_value=0)
    grid.to_csv(sec / "acc_grid_conf_bucket_by_final.csv", encoding="utf-8-sig")
    plt.figure(figsize=(7, 4))
    sns.heatmap(grid, annot=True, fmt=".1f", cmap="YlOrRd")
    plt.title("Accuracy heatmap: confidence_mean bucket × final answer")
    save_plot(sec / "plot_heatmap_acc_by_conf_bucket_answer.png")

# ===================== Section 6: latency =====================
sec = out_root / "6_latency"
sec.mkdir(exist_ok=True)

if "latency_avg_ms" in df.columns and df["latency_avg_ms"].notna().any():
    lat_sum = df["latency_avg_ms"].agg([
        "mean", "median", "min", "max",
        lambda s: np.percentile(s.dropna(), 90),
        lambda s: np.percentile(s.dropna(), 95)
    ]).round(1)
    lat_sum.index = ["mean", "median", "min", "max", "p90", "p95"]
    lat_sum.to_csv(sec / "latency_avg_summary.csv", header=False, encoding="utf-8-sig")

    plt.figure(figsize=(6, 4))
    plt.hist(df["latency_avg_ms"].dropna(), bins=20,
             color="#4E79A7", edgecolor="black")
    plt.xlabel("Latency avg (ms)")
    plt.ylabel("Freq")
    plt.title("Latency avg distribution")
    save_plot(sec / "plot_latency_avg_hist.png")

    bins = [0, 1000, 2000, 3000, 5000, 10000, np.inf]
    labels = ["<=1s", "1-2s", "2-3s", "3-5s", "5-10s", ">10s"]
    df["lat_bucket"] = pd.cut(df["latency_avg_ms"], bins=bins, labels=labels)
    acc_by_lat = df.groupby("lat_bucket")["correct"].mean().mul(100) \
                   .reset_index().rename(columns={"correct": "accuracy_%"})
    acc_by_lat.to_csv(sec / "accuracy_by_latency_bucket.csv", index=False, encoding="utf-8-sig")

    plt.figure(figsize=(7, 4))
    plt.bar(acc_by_lat["lat_bucket"].astype(str),
            acc_by_lat["accuracy_%"], color="#76B7B2")
    plt.ylim(0, 100)
    plt.xlabel("Latency bucket")
    plt.ylabel("Accuracy (%)")
    plt.title("Accuracy vs latency bucket")
    save_plot(sec / "plot_accuracy_by_latency_bucket.png")

if "total_time_ms" in df.columns and df["total_time_ms"].notna().any():
    tm_sum = df["total_time_ms"].agg([
        "mean", "median", "min", "max",
        lambda s: np.percentile(s.dropna(), 90),
        lambda s: np.percentile(s.dropna(), 95)
    ]).round(1)
    tm_sum.index = ["mean", "median", "min", "max", "p90", "p95"]
    tm_sum.to_csv(sec / "total_time_summary.csv", header=False, encoding="utf-8-sig")

    plt.figure(figsize=(6, 4))
    plt.hist(df["total_time_ms"].dropna(), bins=20,
             color="#EDC948", edgecolor="black")
    plt.xlabel("Total time (ms)")
    plt.ylabel("Freq")
    plt.title("Total time distribution")
    save_plot(sec / "plot_total_time_hist.png")

# Scatter latency vs vote_ratio (uncertainty–cost)
if "latency_avg_ms" in df.columns and "vote_ratio_max" in df.columns:
    plt.figure(figsize=(6, 4))
    colors = df["correct"].map({1: "#59A14F", 0: "#E15759"})
    plt.scatter(df["latency_avg_ms"], df["vote_ratio_max"],
                c=colors, alpha=0.6, edgecolors="black", linewidths=0.3)
    plt.xlabel("Latency avg (ms)")
    plt.ylabel("Vote ratio max")
    plt.title("Latency vs vote_ratio_max (colored by correctness)")
    save_plot(sec / "plot_scatter_latency_vs_vote_ratio_colored_correct.png")

# Heatmap accuracy by latency × consensus
if "lat_bucket" in df.columns:
    grid = df.groupby(["lat_bucket", "consensus_label"])["correct"] \
             .mean().mul(100).unstack(fill_value=0)
    grid.to_csv(sec / "acc_grid_latency_by_consensus.csv", encoding="utf-8-sig")
    plt.figure(figsize=(7, 4))
    sns.heatmap(grid, annot=True, fmt=".1f", cmap="GnBu")
    plt.title("Accuracy heatmap: latency bucket × consensus")
    save_plot(sec / "plot_heatmap_acc_by_latency_consensus.png")

# ===================== Section 7: runs variability & diversity =====================
sec = out_root / "7_runs_variability"
sec.mkdir(exist_ok=True)

if conf_cols:
    df["conf_std_runs"]  = df[conf_cols].std(axis=1, skipna=True)
    df["conf_mean_runs"] = df[conf_cols].mean(axis=1, skipna=True)
    df[[model_id_col, "conf_std_runs", "conf_mean_runs", "correct"]] \
      .to_csv(sec / "conf_std_runs.csv", index=False, encoding="utf-8-sig")

    plt.figure(figsize=(6, 4))
    plt.hist(df["conf_std_runs"].dropna(), bins=20,
             color="#59A14F", edgecolor="black")
    plt.xlabel("Std of confidence across runs")
    plt.ylabel("Freq")
    plt.title("Confidence variability across runs")
    save_plot(sec / "plot_conf_std_hist.png")

    plt.figure(figsize=(6, 4))
    sns.boxplot(x=df["correct"], y=df["conf_std_runs"],
                palette=["#E15759", "#59A14F"])
    plt.xticks([0, 1], ["Incorrect", "Correct"])
    plt.ylabel("Conf std across runs")
    plt.title("Conf variability vs correctness")
    save_plot(sec / "plot_conf_std_vs_correct_box.png")

# Diversity of answers across runs
def answer_diversity(row):
    votes = [row.get(c) for c in run_cols] if run_cols else []
    votes = [v for v in votes if v is not None]
    return len(set(votes)) if votes else np.nan

df["answer_diversity"] = df.apply(answer_diversity, axis=1)
div_tbl = df["answer_diversity"].value_counts().rename_axis("unique_answers") \
           .reset_index(name="count")
div_tbl.to_csv(sec / "answer_diversity.csv", index=False, encoding="utf-8-sig")

plt.figure(figsize=(6, 4))
plt.bar(div_tbl["unique_answers"].astype(int).astype(str),
        div_tbl["count"], color="#A0CBE8")
plt.xlabel("#Unique answers across runs")
plt.ylabel("Count")
plt.title("Answer diversity")
save_plot(sec / "plot_answer_diversity_bar.png")

# Boxplot diversity vs correctness
plt.figure(figsize=(6, 4))
sns.boxplot(x=df["correct"], y=df["answer_diversity"],
            palette=["#E15759", "#59A14F"])
plt.xticks([0, 1], ["Incorrect", "Correct"])
plt.ylabel("Answer diversity (#unique options)")
plt.title("Answer diversity vs correctness")
save_plot(sec / "plot_answer_diversity_vs_correct_box.png")

# Flip rate sequential
flip_stats = []
for i in range(1, 5):
    c1, c2 = f"answer_run{i}", f"answer_run{i+1}"
    if c1 in df.columns and c2 in df.columns:
        both = df[[c1, c2]].dropna()
        fr = (both[c1] != both[c2]).mean() * 100 if len(both) > 0 else np.nan
        flip_stats.append({"pair": f"{i}->{i+1}", "flip_rate_%": round(fr, 2)})
flip_df = pd.DataFrame(flip_stats)
flip_df.to_csv(sec / "flip_rate_sequential.csv", index=False, encoding="utf-8-sig")

plt.figure(figsize=(6, 4))
plt.bar(flip_df["pair"], flip_df["flip_rate_%"], color="#E15759")
plt.ylim(0, 100)
plt.xlabel("Run pair")
plt.ylabel("Flip rate (%)")
plt.title("Sequential flip rate")
save_plot(sec / "plot_flip_rate_runs.png")

# ===================== Section 8: explanations (length stats) =====================
sec = out_root / "8_explanations"
sec.mkdir(exist_ok=True)

exp_cols = [f"explanation_run{i}" for i in range(1, 6)
            if f"explanation_run{i}" in df.columns]

if exp_cols:
    for i, c in enumerate(exp_cols, start=1):
        df[f"explain_len_words_run{i}"] = df[c].fillna("").astype(str) \
                                            .apply(lambda s: len(str(s).split()))
    long_cols = [c for c in df.columns if c.startswith("explain_len_words_run")]

    df["explain_len_words_mean"] = df[long_cols].mean(axis=1, skipna=True)

    len_stats = df[long_cols + ["explain_len_words_mean", "correct"]]
    len_stats.to_csv(sec / "explain_lengths_runs.csv", index=False, encoding="utf-8-sig")

    plt.figure(figsize=(6, 4))
    sns.boxplot(x=df["correct"], y=df["explain_len_words_mean"],
                palette=["#E15759", "#59A14F"])
    plt.xticks([0, 1], ["Incorrect", "Correct"])
    plt.ylabel("Mean explanation length (words)")
    plt.title("Mean explanation length vs correctness")
    save_plot(sec / "plot_explain_len_mean_vs_correct_box.png")

    # per run
    for i in range(1, 6):
        c = f"explain_len_words_run{i}"
        if c in df.columns:
            plt.figure(figsize=(6, 4))
            sns.boxplot(x=df["correct"], y=df[c],
                        palette=["#E15759", "#59A14F"])
            plt.xticks([0, 1], ["Incorrect", "Correct"])
            plt.ylabel(f"Explanation length run{i} (words)")
            plt.title(f"Explanation length vs correctness (run{i})")
            save_plot(sec / f"plot_explain_len_run{i}_vs_correct_box.png")

    # scatter mean length vs confidence_mean
    if "confidence_mean" in df.columns:
        plt.figure(figsize=(6, 4))
        colors = df["correct"].map({1: "#59A14F", 0: "#E15759"})
        plt.scatter(df["explain_len_words_mean"], df["confidence_mean"],
                    c=colors, alpha=0.6, edgecolors="black", linewidths=0.3)
        plt.xlabel("Mean explanation length (words)")
        plt.ylabel("confidence_mean")
        plt.title("Explain length vs confidence (colored by correctness)")
        save_plot(sec / "plot_scatter_explain_len_mean_vs_conf_mean_colored_correct.png")

    # heatmap accuracy by exp_len bucket × final answer
    bins = [0, 10, 20, 40, 80, 160, np.inf]
    labels = ["<=10", "10-20", "20-40", "40-80", "80-160", ">160"]
    df["exp_len_bucket"] = pd.cut(df["explain_len_words_mean"], bins=bins, labels=labels)
    grid = df.groupby(["exp_len_bucket", "answer_final"])["correct"] \
             .mean().mul(100).unstack(fill_value=0)
    grid.to_csv(sec / "acc_grid_exp_len_bucket_by_final.csv", encoding="utf-8-sig")
    plt.figure(figsize=(7, 4))
    sns.heatmap(grid, annot=True, fmt=".1f", cmap="PuBuGn")
    plt.title("Accuracy heatmap: explanation length × final answer")
    save_plot(sec / "plot_heatmap_acc_by_explain_len_bucket_answer.png")

    # «وقتی درست بوده کوتاه‌تر؟»
    test_tbl = []
    for i in range(1, 6):
        c = f"explain_len_words_run{i}"
        if c in df.columns:
            mu_corr = df.loc[df["correct"] == 1, c].dropna().mean()
            mu_inc  = df.loc[df["correct"] == 0, c].dropna().mean()
            test_tbl.append({
                "run": f"run{i}",
                "mean_len_correct": round(mu_corr, 2),
                "mean_len_incorrect": round(mu_inc, 2),
                "delta_inc_minus_corr": round((mu_inc - mu_corr), 2)
            })
    pd.DataFrame(test_tbl).to_csv(
        sec / "explain_len_correct_vs_incorrect_summary.csv",
        index=False, encoding="utf-8-sig"
    )

# ===================== Section 9: error inspection =====================
sec = out_root / "9_error_inspection"
sec.mkdir(exist_ok=True)

top_mist = pd.crosstab(df["answer_final"], df["gold_primary"]).stack() \
           .reset_index(name="count")
top_mist = top_mist[top_mist["answer_final"] != top_mist["gold_primary"]] \
           .sort_values("count", ascending=False)
top_mist.head(30).to_csv(sec / "top_mistakes_final.csv",
                         index=False, encoding="utf-8-sig")

plt.figure(figsize=(8, 4))
head = top_mist.head(12)
labels = head["answer_final"].astype(str) + "→" + head["gold_primary"].astype(str)
plt.bar(labels, head["count"], color="#E15759")
plt.ylabel("Count")
plt.title("Top mistakes (final→gold)")
plt.xticks(rotation=45, ha="right")
save_plot(sec / "plot_top_mistakes_final_bar.png")

# mistakes per run
for i in range(1, 6):
    c = f"answer_run{i}"
    if c in df.columns:
        tm = pd.crosstab(df[c], df["gold_primary"]).stack() \
               .reset_index(name="count")
        tm = tm[tm[c] != tm["gold_primary"]].sort_values("count", ascending=False)
        tm.head(30).to_csv(sec / f"top_mistakes_run{i}.csv",
                           index=False, encoding="utf-8-sig")
        plt.figure(figsize=(8, 4))
        head = tm.head(12)
        labels = head[c].astype(str) + "→" + head["gold_primary"].astype(str)
        plt.bar(labels, head["count"], color="#B07AA1")
        plt.ylabel("Count")
        plt.title(f"Top mistakes (run{i}→gold)")
        plt.xticks(rotation=45, ha="right")
        save_plot(sec / f"plot_top_mistakes_run{i}_bar.png")

# ===================== Section 10: policy simulation (accuracy–coverage–cost) =====================
sec = out_root / "10_ablations"
sec.mkdir(exist_ok=True)

policies = []
k_thresholds  = [1, 2, 3, 4, 5] if "consensus_max" in df.columns else [None]
vr_thresholds = [0.0, 0.6, 0.7, 0.8, 0.9] if "vote_ratio_max" in df.columns else [None]
cm_thresholds = []
if "confidence_mean" in df.columns:
    # چند threshold نمونه از روی کمینه و بیشینه
    cms = np.linspace(df["confidence_mean"].min(),
                      df["confidence_mean"].max(), 5)
    cm_thresholds = list(cms)
else:
    cm_thresholds = [None]

for k in k_thresholds:
    for vr in vr_thresholds:
        for cm in cm_thresholds:
            sel = pd.Series([True] * len(df))
            name = []
            if k is not None:
                sel &= (df["consensus_max"] >= k)
                name.append(f"k>={k}")
            if vr is not None:
                sel &= (df["vote_ratio_max"] >= vr)
                name.append(f"vr>={vr:.2f}")
            if cm is not None:
                sel &= (df["confidence_mean"] >= cm)
                name.append(f"cm>={cm:.1f}")
            name = " & ".join(name) if name else "all"
            sub = df[sel]
            coverage = len(sub) / len(df) * 100 if len(df) else 0.0
            acc_p = sub["correct"].mean() * 100 if len(sub) else np.nan
            lat_avg = sub["latency_avg_ms"].mean() \
                      if "latency_avg_ms" in sub.columns and len(sub) > 0 else np.nan
            policies.append({
                "policy": name,
                "n": len(sub),
                "coverage_%": coverage,
                "accuracy_%": acc_p,
                "avg_latency_ms": lat_avg
            })

pol_df = pd.DataFrame(policies).sort_values(
    ["accuracy_%", "coverage_%"], ascending=[False, False]
)
pol_df.to_csv(sec / "policy_simulation.csv", index=False, encoding="utf-8-sig")

plt.figure(figsize=(6, 4))
plt.scatter(pol_df["coverage_%"], pol_df["accuracy_%"],
            c="#4E79A7", edgecolors="black", linewidths=0.3)
plt.xlabel("Coverage (%)")
plt.ylabel("Accuracy (%)")
plt.title("Policy frontier (accuracy vs coverage)")
save_plot(sec / "plot_policy_frontier_accuracy_coverage.png")

print(f"✅ All reports saved under: {out_root.resolve()}")


C:\Users\sazgar\AppData\Local\Temp\ipykernel_25208\3546212626.py:328: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("vote_ratio_bucket")["correct"].mean().mul(100)
C:\Users\sazgar\AppData\Local\Temp\ipykernel_25208\3546212626.py:371: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  rel_vr = df.groupby("vote_ratio_bin")["correct"].mean().mul(100) \
C:\Users\sazgar\AppData\Local\Temp\ipykernel_25208\3546212626.py:396: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to ado

✅ All reports saved under: F:\eval_sc
